# 06. Interactive Visualization and Dashboards

**Aim:** Develop interactive charts using Plotly.

## Theory

Interactive visualizations support exploration by allowing hover details, zooming, filtering, and richer tooltips. Compared with static plotting libraries, Plotly is especially useful for dashboards and presentations because users can inspect specific data points and categories in more detail without changing the code.

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../logs/.mplconfig'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

TRAIN_DIR = Path('../data/PlantDoc-Dataset-master/train/')
TEST_DIR = Path('../data/PlantDoc-Dataset-master/test/')
MANIFEST_PATH = Path('../data/dataset_manifest.csv')
CLEAN_MANIFEST_PATH = Path('../data/dataset_manifest_clean.csv')
FEATURES_PATH = Path('../data/image_features.csv')

def scan_split(split_dir: Path, split_name: str) -> pd.DataFrame:
    records = []
    if not split_dir.exists():
        return pd.DataFrame(columns=['image_path', 'class_name', 'split'])
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file():
                records.append({
                    'image_path': str(image_path.as_posix()),
                    'class_name': class_dir.name,
                    'split': split_name,
                })
    return pd.DataFrame(records)


def ensure_manifest(prefer_clean: bool = True) -> pd.DataFrame:
    target = CLEAN_MANIFEST_PATH if prefer_clean and CLEAN_MANIFEST_PATH.exists() else MANIFEST_PATH
    if target.exists():
        return pd.read_csv(target)

    train_df = scan_split(TRAIN_DIR, 'train')
    test_df = scan_split(TEST_DIR, 'test')
    full_df = pd.concat([train_df, test_df], ignore_index=True)
    full_df['class_name'] = full_df['class_name'].astype(str).str.strip()
    full_df = full_df.drop_duplicates(subset=['image_path']).reset_index(drop=True)
    return full_df


def validate_image(image_path: str):
    path = Path(image_path)
    if not path.exists():
        return False, 'missing'
    try:
        with Image.open(path) as img:
            img.verify()
        return True, 'ok'
    except Exception:
        return False, 'corrupt'


def load_rgb_image(image_path: str, size=(224, 224)):
    with Image.open(image_path) as img:
        rgb = img.convert('RGB').resize(size)
        return np.array(rgb)


def extract_rgb_features(image_array: np.ndarray) -> dict:
    channels = image_array.reshape(-1, 3)
    return {
        'mean_r': float(channels[:, 0].mean()),
        'mean_g': float(channels[:, 1].mean()),
        'mean_b': float(channels[:, 2].mean()),
        'std_r': float(channels[:, 0].std()),
        'std_g': float(channels[:, 1].std()),
        'std_b': float(channels[:, 2].std()),
    }


def ensure_features() -> pd.DataFrame:
    if FEATURES_PATH.exists():
        return pd.read_csv(FEATURES_PATH)

    manifest_df = ensure_manifest(prefer_clean=True).copy()
    quality_records = []
    for image_path in manifest_df['image_path']:
        valid, status = validate_image(image_path)
        quality_records.append((valid, status))
    manifest_df[['is_valid_image', 'file_status']] = pd.DataFrame(quality_records, index=manifest_df.index)
    manifest_df = manifest_df[manifest_df['is_valid_image']].copy().reset_index(drop=True)

    feature_rows = []
    for row in manifest_df.itertuples(index=False):
        image_array = load_rgb_image(row.image_path)
        feature_rows.append({
            'image_path': row.image_path,
            'class_name': row.class_name,
            'split': row.split,
            **extract_rgb_features(image_array),
        })

    features_df = pd.DataFrame(feature_rows)
    return features_df


## Code

In [2]:
import importlib.util
import subprocess

if importlib.util.find_spec('plotly') is None:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])
        print('plotly installed successfully.')
    except Exception as exc:
        raise RuntimeError('plotly is required for this notebook. Installation failed.') from exc

import plotly.express as px
import plotly.graph_objects as go

manifest_df = ensure_manifest(prefer_clean=True).copy()
features_df = ensure_features().copy()

class_counts = manifest_df['class_name'].value_counts().reset_index()
class_counts.columns = ['class_name', 'image_count']
fig_bar = px.bar(class_counts, x='class_name', y='image_count', title='Class Image Counts', hover_data=['image_count'])
fig_bar.update_layout(xaxis_tickangle=-75)
fig_bar.show()

fig_scatter = px.scatter(features_df, x='mean_r', y='mean_g', color='class_name', hover_data=['image_path'], title='mean_r vs mean_g')
fig_scatter.show()

top_10_classes = features_df['class_name'].value_counts().head(10).index
fig_box = px.box(features_df[features_df['class_name'].isin(top_10_classes)], x='class_name', y='mean_r', color='class_name', title='mean_r per Class (Top 10)')
fig_box.update_layout(xaxis_tickangle=-75, showlegend=False)
fig_box.show()

corr_matrix = features_df[['mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b']].corr()
fig_heatmap = go.Figure(data=go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index, colorscale='RdBu', zmid=0))
fig_heatmap.update_layout(title='Feature Correlation Heatmap')
fig_heatmap.show()

split_counts = manifest_df['split'].value_counts().reset_index()
split_counts.columns = ['split', 'image_count']
fig_pie = px.pie(split_counts, names='split', values='image_count', title='Train vs Test Split')
fig_pie.show()

## Results & Evaluation

In [3]:
print('Interactive Plotly charts rendered: 5')
print('Charts included: bar, scatter, box, heatmap, pie')

Interactive Plotly charts rendered: 5
Charts included: bar, scatter, box, heatmap, pie


## Conclusion

Plotly adds an exploratory layer to the PlantDoc analysis by making distributions, feature relationships, and split proportions interactive. This is especially useful when presenting findings or drilling down into specific classes and samples.